# Imports

In [1]:
from keras.models import Sequential
from keras.layers import Dense, Flatten, Dropout, Conv2D, MaxPooling2D
import pathlib
import keras
import keras_tuner

import tensorflow as tf

## Download dataset

In [2]:

DATASET_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
DATASET_NAME = "flower_photos"
CACHE_DIR = "."

data_dir = pathlib.Path(
    tf.keras.utils.get_file(
        origin    = DATASET_URL,
        fname     = DATASET_NAME,   # nome del file .tgz salvato
        cache_dir = CACHE_DIR,
        untar     = True,              # estrae automaticamente
        extract   = True,
    )
)

### Parameters

In [3]:
IMG_HEIGHT = 180
IMG_WIDTH  = 180
BATCH_SIZE = 32
SEED       = 123

VAL_SPLIT  = 0.2
AUTOTUNE = tf.data.AUTOTUNE

### Util functions

In [4]:
def configure_for_performance(ds):
  ds = ds.cache()
  ds = ds.shuffle(buffer_size=1000)
  ds = ds.batch(BATCH_SIZE)
  ds = ds.prefetch(buffer_size=AUTOTUNE)
  return ds

In [5]:
train_ds = keras.utils.image_dataset_from_directory(
  data_dir / DATASET_NAME,
  validation_split=VAL_SPLIT,
  subset="training",
  seed=SEED,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE)

val_ds = keras.utils.image_dataset_from_directory(
  data_dir / DATASET_NAME,
  validation_split=VAL_SPLIT,
  subset="validation",
  seed=SEED,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE)

# train_ds = configure_for_performance(train_ds)
# val_ds = configure_for_performance(val_ds)

Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.


### Layers

In [ ]:
def build_model(hp):
    model = Sequential()
    model.add(tf.keras.layers.Rescaling(1./255))
    model.add(Conv2D(hp.Choice('units_1', [8, 16, 32]), hp.Choice('kernel_1', [3, 5, 7]), activation=hp.Choice('act_1', ['relu', 'leaky_relu'])))
    model.add(MaxPooling2D())
    model.add(Conv2D(hp.Choice('units_2', [8, 16, 32]), hp.Choice('kernel_2', [3, 5, 7]), activation=hp.Choice('act_2', ['relu', 'leaky_relu'])))
    model.add(MaxPooling2D())
    model.add(Conv2D(hp.Choice('units_3', [8, 16, 32]), hp.Choice('kernel_3', [3, 5, 7]), activation=hp.Choice('act_3', ['relu', 'leaky_relu'])))
    model.add(MaxPooling2D())
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(5))

    model.compile(
        optimizer='adam',
        loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy'])

    return model

In [7]:
#model.fit(
#  train_ds,
#  validation_data=val_ds,
#  epochs=10
#)

In [13]:
tuner = keras_tuner.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=10)

In [15]:
tuner.search(train_ds, epochs=5, validation_data=val_ds)
best_model = tuner.get_best_models()[0]

FatalTypeError: Expected the model-building function, or HyperModel.build() to return a valid Keras Model instance. Received: None of type <class 'NoneType'>.